# Email → Product Feedback Sentiment Analyzer

Build an event-driven feedback pipeline: incoming emails trigger real-time
sentiment analysis in a sandbox-hosted Flask app.

```
Customer Email → Outlook Inbox → Gateway (trigger config) → Sandbox (Flask sentiment analyzer)
```

## Prerequisites
- Azure CLI [signed in](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)
- SDKs: `pip install azure-connectorgateway azure-sandbox azure-mgmt-sandbox`

Click **Run All** or go **Step by Step**.

### 0. Initialize

In [ ]:
import os, sys, json, subprocess, time
from azure.connectorgateway import ConnectorGatewayClient, TriggerClient
from azure.sandbox import SandboxClient
from azure.mgmt.sandbox import SandboxGroupManagementClient

lab_name = os.path.basename(os.path.dirname(os.path.abspath(
    globals().get('__vsc_ipynb_file__', os.path.join(os.getcwd(), '__file__')))))

_SHELL = sys.platform == 'win32'
account = json.loads(subprocess.run(
    ['az', 'account', 'show', '-o', 'json'],
    capture_output=True, text=True, check=True, shell=_SHELL).stdout)

subscription_id = account['id']
resource_group_name = f'lab-{lab_name}'
resource_group_location = 'eastus2'
sandbox_group_name = f'{lab_name}-sg'
gateway_name = f'{lab_name}-gw'
connection_name = 'o365-trigger-conn'
trigger_config_name = 'email-dashboard-trigger'

trigger_client = TriggerClient(subscription_id=subscription_id, resource_group=resource_group_name)
conn_client = ConnectorGatewayClient(subscription_id=subscription_id, resource_group=resource_group_name)
sbx_client = SandboxClient(subscription_id=subscription_id, resource_group=resource_group_name)
mgmt = SandboxGroupManagementClient(subscription_id=subscription_id, resource_group=resource_group_name)

print(f'Lab:            {lab_name}')
print(f'User:           {account["user"]["name"]}')
print(f'Subscription:   {account["name"]} ({subscription_id})')
print(f'Resource Group: {resource_group_name}')
print(f'Gateway:        {gateway_name}')
print(f'Sandbox Group:  {sandbox_group_name}')

### 1. Create resource group and connector gateway

The gateway needs a SystemAssigned managed identity to authenticate callbacks to the sandbox.

In [ ]:
!az group create --name {resource_group_name} --location {resource_group_location} -o none

try:
    gw = conn_client.get_gateway(gateway_name)
    print(f'Gateway: {gateway_name} (exists)')
except Exception:
    gw = conn_client.create_gateway(gateway_name, location='brazilsouth',
        identity={'type': 'SystemAssigned'})
    print(f'Gateway: {gateway_name} (created)')

gw_principal_id = gw['identity']['principalId']
gw_tenant_id = gw['identity']['tenantId']
print(f'Principal ID: {gw_principal_id}')

### 2. Create OAuth connection

Creates an Office 365 connection on the gateway. If not yet authorized,
click the consent link immediately — it expires quickly.

In [ ]:
try:
    conn = conn_client.create_connection(gateway_name, connection_name,
        connector_name='office365')
    print(f'Connection: {connection_name} (created)')
except Exception as e:
    if '409' in str(e) or 'Conflict' in str(e):
        print(f'Connection: {connection_name} (already exists)')
    else:
        raise

conn = conn_client.get_connection(gateway_name, connection_name)
status = conn.get('properties', {}).get('statuses', [{}])[0].get('status', 'Unknown')
print(f'Status: {status}')

if status != 'Connected':
    link = conn_client.generate_consent_link(gateway_name, connection_name)
    print(f'\n⚠️  Authorize here (click immediately):\n{link}')
else:
    print('✅ Already connected!')

### 3. Create sandbox

In [ ]:
group = mgmt.create_group(sandbox_group_name, location=resource_group_location)
print(f'Sandbox group: {group["name"]}')

sandboxes = sbx_client.list_sandboxes(sandbox_group_name)
if sandboxes:
    sandbox_id = sandboxes[0]['id']
    print(f'Sandbox: {sandbox_id} (existing)')
else:
    sbx = sbx_client.create_sandbox(sandbox_group_name, disk='ubuntu')
    sandbox_id = sbx['id']
    print(f'Sandbox: {sandbox_id} (created)')

# Wait for Running state
print('Waiting for sandbox...')
resumed = False
for i in range(36):
    info = sbx_client.get_sandbox(sandbox_id, sandbox_group_name)
    state = info.get('state', 'Unknown')
    print(f'  [{i+1}] state={state}')
    if state == 'Running':
        break
    if state == 'Idle' and not resumed:
        sbx_client.resume_sandbox(sandbox_id, sandbox_group_name)
        resumed = True
    time.sleep(5)
else:
    raise RuntimeError('Sandbox did not reach Running state in 3 minutes')

print(f'✅ Sandbox is running!')

### 4. Install dependencies

Install Flask in the sandbox (skipped if already present).

In [ ]:
r = sbx_client.exec(sandbox_id, sandbox_group_name,
    'dpkg -s python3-flask >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq python3-flask 2>&1 | tail -1)')
print(f'Flask: {r["stdout"].strip() or "already installed"}')

### 5. Deploy the Product Feedback Analyzer

A Flask app with keyword-based sentiment analysis:
- **POST /webhook** — receives trigger payloads, classifies as positive/negative/neutral
- **GET /** — dashboard with sentiment stats and color-coded email cards
- **GET /email/\<id\>** — full email content

In [ ]:
app_code = """\
from flask import Flask, request, jsonify
import json, datetime, os, re, html as html_lib

app = Flask(__name__)
emails = []

POSITIVE_WORDS = {"love", "great", "excellent", "amazing", "fantastic", "wonderful",
    "happy", "satisfied", "awesome", "brilliant", "perfect", "impressed",
    "recommend", "intuitive", "fast", "reliable", "easy", "helpful", "best",
    "good", "nice", "enjoy", "smooth", "responsive", "pleased", "delighted"}
NEGATIVE_WORDS = {"hate", "terrible", "awful", "horrible", "bad", "worst",
    "broken", "slow", "crash", "bug", "frustrating", "disappointed",
    "unusable", "confusing", "annoying", "poor", "fail", "failed", "error",
    "difficult", "expensive", "waste", "ugly", "laggy", "unresponsive", "negative",
    "concern", "concerns", "issue", "issues", "problem", "problems", "needed",
    "lacking", "missing", "inadequate", "complaint", "dissatisfied", "unhappy"}
NEGATORS = {"not", "no", "never", "don", "doesn", "didn", "wasn", "isn",
    "won", "wouldn", "shouldn", "couldn", "hardly", "barely"}
SKIP_PHRASES = {"thank", "thanks", "thank you", "regards", "sincerely"}

def analyze_sentiment(text):
    words = re.findall(r"[a-z]+", text.lower())
    pos, neg = 0, 0
    for i, w in enumerate(words):
        if w in SKIP_PHRASES:
            continue
        negated = i > 0 and words[i - 1] in NEGATORS
        if w in POSITIVE_WORDS:
            if negated: neg += 1
            else: pos += 1
        elif w in NEGATIVE_WORDS:
            if negated: pos += 1
            else: neg += 1
    if pos > neg: return "positive", pos, neg
    elif neg > pos: return "negative", pos, neg
    return "neutral", pos, neg

def strip_html(text):
    return html_lib.unescape(re.sub(r"<[^>]+>", "", text)).strip()

@app.route("/webhook", methods=["POST"])
def webhook():
    payload = request.get_json(silent=True) or {}
    body = payload.get("body", payload)
    if isinstance(body, dict) and "value" in body:
        items = body["value"]
    elif isinstance(body, dict) and "subject" in body:
        items = [body]
    else:
        items = [body] if body else []
    for item in items:
        subject = item.get("subject", "(no subject)")
        body_preview = item.get("bodyPreview", "")
        body_content = item.get("body", body_preview)
        from_addr = item.get("from", "unknown")
        received = item.get("receivedDateTime", datetime.datetime.now(datetime.timezone.utc).isoformat())
        full_text = f"{subject} {strip_html(body_content) if isinstance(body_content, str) else body_preview}"
        sentiment, pos_count, neg_count = analyze_sentiment(full_text)
        emails.append({
            "id": len(emails), "subject": subject, "from": from_addr,
            "received": received, "body_preview": body_preview[:200],
            "body_full": body_content if isinstance(body_content, str) else str(body_content),
            "sentiment": sentiment, "pos_words": pos_count, "neg_words": neg_count,
        })
    return jsonify(status="received", analyzed=len(items), total=len(emails))

@app.route("/")
def dashboard():
    pos = sum(1 for e in emails if e["sentiment"] == "positive")
    neg = sum(1 for e in emails if e["sentiment"] == "negative")
    neu = sum(1 for e in emails if e["sentiment"] == "neutral")
    html = "<html><head><style>"
    html += "body{font-family:sans-serif;max-width:900px;margin:0 auto;padding:20px}"
    html += ".card{border:1px solid #ddd;border-radius:8px;padding:16px;margin:12px 0}"
    html += ".positive{border-left:5px solid #28a745} .negative{border-left:5px solid #dc3545} .neutral{border-left:5px solid #6c757d}"
    html += ".stats{display:flex;gap:20px;margin:16px 0} .stat{padding:12px 24px;border-radius:8px;text-align:center}"
    html += ".stat-pos{background:#d4edda;color:#155724} .stat-neg{background:#f8d7da;color:#721c24} .stat-neu{background:#e2e3e5;color:#383d41}"
    html += "a{color:#0366d6;text-decoration:none}"
    html += "</style></head><body>"
    html += "<h1>&#128231; Product Feedback Analyzer</h1>"
    html += "<p>Real-time sentiment analysis on incoming customer feedback emails</p>"
    html += "<div class='stats'>"
    html += f"<div class='stat stat-pos'><b>{pos}</b><br>Positive</div>"
    html += f"<div class='stat stat-neg'><b>{neg}</b><br>Negative</div>"
    html += f"<div class='stat stat-neu'><b>{neu}</b><br>Neutral</div>"
    html += f"<div class='stat' style='background:#e7f3ff'><b>{len(emails)}</b><br>Total</div>"
    html += "</div><hr>"
    if not emails:
        html += "<p><i>No emails received yet. Send a product feedback email to trigger the analysis!</i></p>"
    for e in reversed(emails[-50:]):
        badge = {"positive": "&#9989;", "negative": "&#10060;", "neutral": "&#10134;"}[e["sentiment"]]
        html += f"<div class='card {e['sentiment']}'>"
        html += f"<b>{badge} {e['sentiment'].upper()}</b> (positive:{e['pos_words']} negative:{e['neg_words']})<br>"
        html += f"<b>Subject:</b> {e['subject']}<br>"
        html += f"<b>From:</b> {e['from']} &middot; {e['received'][:19]}<br>"
        html += f"<b>Preview:</b> {e['body_preview'][:150]}<br>"
        html += f"<a href='/email/{e['id']}'>View full email &rarr;</a>"
        html += "</div>"
    html += "</body></html>"
    return html

@app.route("/email/<int:email_id>")
def email_detail(email_id):
    if email_id < 0 or email_id >= len(emails): return "Not found", 404
    e = emails[email_id]
    badge = {"positive": "&#9989;", "negative": "&#10060;", "neutral": "&#10134;"}[e["sentiment"]]
    html = "<html><head><style>body{font-family:sans-serif;max-width:800px;margin:0 auto;padding:20px}</style></head><body>"
    html += "<a href='/'>&larr; Back to Dashboard</a><hr>"
    html += f"<h2>{badge} {e['subject']}</h2>"
    html += f"<p><b>From:</b> {e['from']}<br><b>Received:</b> {e['received']}<br>"
    html += f"<b>Sentiment:</b> {e['sentiment'].upper()} (positive words: {e['pos_words']}, negative words: {e['neg_words']})</p>"
    html += "<hr><h3>Full Email Content</h3>"
    html += f"<div style='border:1px solid #eee;padding:16px;border-radius:8px'>{e['body_full']}</div>"
    html += "</body></html>"
    return html

@app.route("/api/emails")
def api_emails():
    return jsonify(emails=emails, stats={"positive": sum(1 for e in emails if e["sentiment"]=="positive"), "negative": sum(1 for e in emails if e["sentiment"]=="negative"), "neutral": sum(1 for e in emails if e["sentiment"]=="neutral")})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
"""

sbx_client.exec(sandbox_id, sandbox_group_name, 'pkill -f server.py 2>/dev/null; sleep 1; true')
sbx_client.write_file(sandbox_id, sandbox_group_name, '/app/server.py', app_code.strip())
print('Uploaded /app/server.py')

sbx_client.exec(sandbox_id, sandbox_group_name, 'nohup python3 /app/server.py > /tmp/server.log 2>&1 &')
time.sleep(2)
r = sbx_client.exec(sandbox_id, sandbox_group_name, 'curl -s http://localhost:5000/ | head -c 100')
print(f'Server: {"running" if "Feedback" in r["stdout"] else "starting..."}')

try:
    sbx_client.add_port(sandbox_id, sandbox_group_name, 5000, anonymous=True)
except Exception:
    pass

dashboard_url = f'https://{sandbox_id}--5000.proxy.azuredevcompute.io'
print(f'\nDashboard: {dashboard_url}')

### 6. Discover trigger operations

Lists available trigger operations from the Office 365 connector's Swagger definition.

In [ ]:
try:
    ops = trigger_client.list_trigger_operations(gateway_name, 'office365')
    print(f'Found {len(ops)} trigger operations for office365:\n')
    for op in ops:
        print(f'  • {op["operationId"]}: {op.get("summary", "")}')
        print(f'    Type: {op.get("triggerType", "unknown")}')
except Exception as e:
    print(f'Could not discover operations: {e}')
    print('\nUsing known operation: OnNewEmailV3 (Notification trigger)')

### 7. Create trigger config + access policy

Creates the trigger config on the gateway and grants the gateway's MI access
to invoke the sandbox port. Safe to re-run — deletes and recreates if exists.

In [ ]:
try:
    trigger_client.delete_trigger(gateway_name, trigger_config_name)
    print(f'Deleted existing trigger config: {trigger_config_name}')
except Exception:
    pass

trigger = trigger_client.create_trigger(
    gateway_name, trigger_config_name,
    connector_name='office365',
    connection_name=connection_name,
    operation_name='OnNewEmailV3',
    sandbox_id=sandbox_id,
    sandbox_group=sandbox_group_name,
    port=5000,
    port_path='/webhook',
    http_method='POST',
    description='Product feedback analyzer - sentiment analysis on incoming emails',
    parameters=[{'name': 'folderPath', 'value': 'Inbox'}, {'name': 'subjectFilter', 'value': 'Feedback'}],
)
print(f'✅ Trigger config created: {trigger_config_name}')
print(f'State: {trigger["properties"]["state"]}')
print(f'Callback: {trigger["properties"]["notificationDetails"]["callbackUrl"]}')

try:
    conn_client.create_access_policy(
        gateway_name, connection_name,
        principal_id=gw_principal_id,
        tenant_id=gw_tenant_id,
        location='brazilsouth')
    print(f'✅ Access policy granted for gateway MI')
except Exception as e:
    if '409' in str(e) or 'Conflict' in str(e):
        print('✅ Access policy already exists')
    else:
        raise

### 8. Verify trigger is active

In [ ]:
tc = trigger_client.get_trigger(gateway_name, trigger_config_name)
state = tc['properties']['state']
print(f'Trigger config state: {state}')
if state == 'Enabled':
    print('✅ Trigger is active and listening for events!')
else:
    print(f'⚠️  State is {state} — may take a moment to activate')

### 9. Test it — send yourself an email

The trigger is configured with a **subject filter** — only emails whose subject
contains **"Feedback"** will be processed.

Send yourself an email (with "Feedback" in the subject) and wait 30–60 seconds
for the trigger to fire. Then run the cell below.

**Sample emails to send:**

✅ Positive: `Subject: Feedback - Amazing product experience!`

❌ Negative: `Subject: Feedback - Frustrated with recent update`

➖ Neutral: `Subject: Feedback - Question about pricing tiers`

In [ ]:
r = sbx_client.exec(sandbox_id, sandbox_group_name, 'curl -s http://localhost:5000/api/emails')
data = json.loads(r['stdout'])
stats = data['stats']
print(f'Emails analyzed: {len(data["emails"])}')
print(f'  Positive: {stats["positive"]}  Negative: {stats["negative"]}  Neutral: {stats["neutral"]}')
print(f'\nDashboard: {dashboard_url}')
for e in data['emails'][-5:]:
    badge = {'positive': '✅', 'negative': '❌', 'neutral': '➖'}[e['sentiment']]
    print(f'  {badge} [{e["sentiment"]}] {e["subject"]}')

### 10. Lifecycle management

In [ ]:
result = trigger_client.disable_trigger(gateway_name, trigger_config_name)
print(f'⏸️  Disabled: state={result["properties"]["state"]}')
time.sleep(2)

result = trigger_client.enable_trigger(gateway_name, trigger_config_name)
print(f'▶️  Enabled: state={result["properties"]["state"]}')

print('\n--- Full trigger config ---')
detail = trigger_client.get_trigger(gateway_name, trigger_config_name)
print(json.dumps(detail, indent=2, default=str))

### 11. Clean up

In [ ]:
trigger_client.delete_trigger(gateway_name, trigger_config_name)
print(f'Deleted trigger: {trigger_config_name}')

sbx_client.delete_sandbox(sandbox_id, sandbox_group_name)
print(f'Deleted sandbox')

mgmt.delete_group(sandbox_group_name)
print(f'Deleted sandbox group')

conn_client.delete_gateway(gateway_name)
print(f'Deleted gateway')

!az group delete --name {resource_group_name} --yes --no-wait
print(f'Deleting resource group')